# Rank-one matrix completion with ROL

**SIAM MDS 2026 minitutorial, exercise A.**

You will solve the Quick Start problem with ROL's Python interface, using exactly the code shown on the slides: a `NumPyVector` for the variable, an `Objective`, and a `Problem`/`Solver` pair. Then you will explore how the starting point and side information change the answer.

In [ ]:
%pip install -q --only-binary=:all: "rol-python==2026.11.9.dev1945"

In [ ]:
# ROL prints from C++. Colab needs wurlitzer to show that output in the notebook;
# other Jupyter kernels (e.g. Binder) show it already, and wurlitzer would hide it there.
import sys
if 'google.colab' in sys.modules:
    %pip install -q wurlitzer
    %load_ext wurlitzer

In [ ]:
import sys
from importlib.metadata import version

import numpy as np
from pyrol import Objective, Constraint, Problem, Solver, Bounds, getCout
from pyrol.pyrol.Teuchos import ParameterList

print('Python', sys.version.split()[0], '| rol-python', version('rol-python'))

## 1. The optimization variable is a ROL `Vector`

ROL's algorithms combine vectors only through methods such as `plus`, `scale`, `dot` and `clone`. This class implements them for a NumPy array; `x.array[:4]` holds $u$ and `x.array[4:]` holds $v$. Note that `clone()` returns a new zero vector, not a copy. The inherited `dual()` returns the vector itself, which is correct for this Euclidean `dot`; it shares storage with `x`, so never use it in place of `clone()`.

In [ ]:
import numpy as np
from pyrol.pyrol import ROL
from pyrol.getTypeName import getTypeName

class NumPyVector(getTypeName('Vector')):
    def __init__(self, array=None):
        assert isinstance(array, np.ndarray)
        self.array = array
        super().__init__()

    def plus(self, b):
        self.array += b.array

    def scale(self, alpha):
        self.array *= alpha

    def dot(self, b):
        return np.vdot(self.array, b.array)

    def norm(self):
        return self.dot(self)**0.5

    def zero(self):
        self.setScalar(0.)

    def clone(self):
        array = np.empty_like(self.array)
        ans = NumPyVector(array)
        ans.zero()
        return ans

    def axpy(self, alpha, x):
        self.array += alpha*x.array

    def dimension(self):
        return len(self.array)

    def setScalar(self, new_value):
        self.array[:] = new_value

    def reduce(self, op):
        reductionType = op.reductionType()
        if reductionType == ROL.Elementwise.REDUCE_MIN:
            return self.array.min()
        elif reductionType == ROL.Elementwise.REDUCE_MAX:
            return self.array.max()
        elif reductionType == ROL.Elementwise.REDUCE_SUM:
            return self.array.sum()
        elif reductionType == ROL.Elementwise.REDUCE_AND:
            return np.logical_and.reduce(self.array)
        elif reductionType == ROL.Elementwise.REDUCE_BOR:
            return np.bitwise_or.reduce(self.array)
        else:
            raise NotImplementedError(reductionType)

    def applyUnary(self, op):
        for i in range(self.dimension()):
            self.array[i] = op.apply(self.array[i])

    def applyBinary(self, op, other):
        assert self.dimension() == other.dimension()
        for i in range(self.dimension()):
            self.array[i] = op.apply(self.array[i], other.array[i])

    def __getitem__(self, index):
        return self.array[index]

    def __setitem__(self, index, val):
        self.array[index] = val

    def basis(self, i):
        b = self.clone()
        b.array[:] = 0.
        b.array[i] = 1.
        self._basis = b
        return b

## 2. The problem

Only the nine entries of $Y\in\mathbb R^{4\times4}$ in the set $\Omega$ (mask $M$) are observed. We minimize over the factors $x=(u,v)\in\mathbb R^8$

$$F(x)=\tfrac12\lVert M\odot(uv^\top-Y)\rVert_F^2+\tfrac{\gamma}{2}\bigl(\lVert u\rVert^2+\lVert v\rVert^2\bigr),\qquad\gamma=0.1.$$

With $R=M\odot(uv^\top-Y)$ and a direction $d=(\delta u,\delta v)$: $\nabla F(x)=(Rv+\gamma u,\;R^\top u+\gamma v)$ and $\nabla^2F(x)\,d=(\delta R\,v+R\,\delta v+\gamma\,\delta u,\;\delta R^\top u+R^\top\delta u+\gamma\,\delta v)$ with $\delta R=M\odot(\delta u\,v^\top+u\,\delta v^\top)$.

In [ ]:
# Data: Y has rank one (every entry is 2); M is the 0/1 mask of the observed set.
Y = np.full((4, 4), 2.0)
M = np.zeros((4, 4))
M[:2, :2] = M[2:, 2:] = 1.0
M[1, 2] = 1.0
gamma = 0.1
total = 32.0    # sum of all 16 entries of Y, used by the constrained variant

In [ ]:
class RankOneMatrixObjective(Objective):
    def value(self, x, tol):
        u, v = x.array[:4], x.array[4:]
        R = M * (np.outer(u, v) - Y)
        return 0.5*np.sum(R**2) + 0.5*gamma*(u @ u + v @ v)

    def gradient(self, g, x, tol):
        u, v = x.array[:4], x.array[4:]
        R = M * (np.outer(u, v) - Y)
        g.array[:4] = R @ v + gamma*u
        g.array[4:] = R.T @ u + gamma*v

    def hessVec(self, hv, d, x, tol):
        u, v = x.array[:4], x.array[4:]
        du, dv = d.array[:4], d.array[4:]
        R = M * (np.outer(u, v) - Y)
        dR = M * (np.outer(du, v) + np.outer(u, dv))
        hv.array[:4] = dR @ v + R @ dv + gamma*du
        hv.array[4:] = dR.T @ u + R.T @ du + gamma*dv

**Check the derivatives with ROL.** `checkGradient` and `checkHessVec` compare `gradient` and `hessVec` with finite differences of `value` and `gradient` along a direction $d$, and print a table. In its last column, the error should fall about tenfold with each tenfold smaller step, until rounding errors take over.

In [ ]:
objective = RankOneMatrixObjective()
x = NumPyVector(np.sqrt(2.0)*np.ones(8))
d = NumPyVector(np.random.default_rng(0).standard_normal(8))
stream = getCout()
objective.checkGradient(x, d, True, stream)
objective.checkHessVec(x, d, True, stream)
stream.flush();

## 3. Problem and solver

A `Problem` combines the objective with the vector that holds the initial point; the `Solver` is configured by a `ParameterList`. `solve` overwrites `x` in place, so we return a copy.

In [ ]:
def parameters():
    params = ParameterList()
    params['General'] = ParameterList()
    params['General']['Output Level'] = 1
    params['Step'] = ParameterList()
    params['Step']['Type'] = 'Trust Region'
    params['Step']['Trust Region'] = ParameterList()
    params['Step']['Trust Region']['Subproblem Solver'] = 'Truncated CG'
    params['Step']['Trust Region']['Initial Radius'] = 1.0
    params['Status Test'] = ParameterList()
    params['Status Test']['Gradient Tolerance'] = 1e-8
    params['Status Test']['Iteration Limit'] = 120
    return params

def solve(x0):
    x = NumPyVector(np.array(x0, dtype=float))
    objective = RankOneMatrixObjective()
    problem = Problem(objective, x)
    solver = Solver(problem, parameters())
    stream = getCout()
    solver.solve(stream)
    return x.array.copy()

In [ ]:
def summary(label, x):
    """Print F, the gradient norm and the missing-entry RMSE at x (in Python)."""
    objective, g = RankOneMatrixObjective(), NumPyVector(np.zeros(8))
    xv = NumPyVector(np.array(x, dtype=float))
    objective.gradient(g, xv, 0.0)
    X = np.outer(x[:4], x[4:])
    rmse = np.sqrt(np.mean((X[M == 0] - Y[M == 0])**2))
    print(f'{label}: F = {objective.value(xv, 0.0):.7f}, '
          f'|grad F| = {np.linalg.norm(g.array):.1e}, missing-entry RMSE = {rmse:.4f}')
    print(np.round(X, 3))

## 4. Two starting points

**Predict first:** if both runs below report convergence, must the completed matrices agree?

In [ ]:
s = np.sqrt(2.0)
x0 = s * np.ones(8)
x0_tilde = s * np.array([1, 1, -1, -1, 1, 1, -1, -1])
x_lo, x_hi = solve(x0), solve(x0_tilde)

In [ ]:
summary('from x0', x_lo)
summary('from x0_tilde', x_hi)

## 5. Explore

Perturb the sign-flipped start. Which minimum does ROL reach? Try other seeds and sizes.

In [ ]:
rng = np.random.default_rng(0)
x_start = x0_tilde + 0.3 * rng.standard_normal(8)
x_new = solve(x_start)

In [ ]:
summary('from a perturbed x0_tilde', x_new)

## 6. Adding constraints

Additional knowledge can exclude the wrong-sign completion: nonnegative factors, $x\ge0$ (a bound constraint), or a known sum of all entries, the equality $c(x)=(\mathbf 1^\top u)(\mathbf 1^\top v)-32=0$, implemented by a `Constraint` with its Jacobian, adjoint and adjoint Hessian. The known-total solve starts from the sign-flipped point $\tilde x_0$ (`x0_tilde`); the bound-constrained solve starts from `max(x0_tilde, 1e-3)`, the projection of `x0_tilde` onto $x\ge10^{-3}$.

In [ ]:
class KnownTotal(Constraint):
    def value(self, c, x, tol):
        u, v = x.array[:4], x.array[4:]
        c[0] = u.sum()*v.sum() - total

    def applyJacobian(self, jv, d, x, tol):
        u, v = x.array[:4], x.array[4:]
        du, dv = d.array[:4], d.array[4:]
        jv[0] = v.sum()*du.sum() + u.sum()*dv.sum()

    def applyAdjointJacobian(self, ajv, lam, x, tol):
        u, v = x.array[:4], x.array[4:]
        ajv.array[:4] = lam[0]*v.sum()
        ajv.array[4:] = lam[0]*u.sum()

    def applyAdjointHessian(self, ahv, lam, d, x, tol):
        du, dv = d.array[:4], d.array[4:]
        ahv.array[:4] = lam[0]*dv.sum()
        ahv.array[4:] = lam[0]*du.sum()

In [ ]:
def augmented_lagrangian_parameters():
    params = ParameterList()
    params['General'] = ParameterList()
    params['General']['Output Level'] = 1
    params['Step'] = ParameterList()
    params['Step']['Type'] = 'Augmented Lagrangian'
    return params

def solve_nonnegative(x0):
    x = NumPyVector(np.array(x0, dtype=float))
    objective = RankOneMatrixObjective()
    problem = Problem(objective, x)
    lower = NumPyVector(np.zeros(8))
    problem.addBoundConstraint(Bounds(lower, True))
    solver = Solver(problem, parameters())
    stream = getCout()
    solver.solve(stream)
    return x.array.copy()

def solve_known_total(x0):
    x = NumPyVector(np.array(x0, dtype=float))
    objective = RankOneMatrixObjective()
    problem = Problem(objective, x)
    lam = NumPyVector(np.zeros(1))
    problem.addConstraint('known total', KnownTotal(), lam)
    solver = Solver(problem, augmented_lagrangian_parameters())
    stream = getCout()
    solver.solve(stream)
    return x.array.copy()

In [ ]:
x_nonneg = solve_nonnegative(np.maximum(x0_tilde, 1e-3))
x_total = solve_known_total(x0_tilde)

In [ ]:
summary('x >= 0, from max(x0_tilde, 1e-3)', x_nonneg)
# Under the equality constraint grad F need not vanish: grad F + c'(x)* w = 0.
summary('known total, from x0_tilde', x_total)
print('constraint residual c(x) =', x_total[:4].sum() * x_total[4:].sum() - total)

## Questions

1. Why do the two unconstrained runs give different completed matrices, and why is this not explained by the symmetry $(u,v)\mapsto(-u,-v)$?
2. How does each constraint exclude the wrong-sign completion?
3. What would change if a second entry linking the two diagonal blocks were observed?